# 03 — Holdout final multi-dia (bloqueado até entrar dado novo)

Este é o único caderno autorizado a declarar se a meta foi atingida. Ele exige dias/episódios nunca usados para criar regras, amostragem da linha do tempo (não só erros), presença e atividade rotuladas separadamente, e pelo menos 20% de dupla anotação com adjudicação de divergências.

Fluxo recomendado: congelar código e catálogo de candidatas → sortear episódios por dia/turno → rotular sem revelar predições → travar o gabarito → gerar predições uma vez → abrir e avaliar. Não ajustar limiar após ver este resultado.

In [1]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
from IPython.display import display

for pasta in (Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks'):
    if (pasta / 'improdutividade_eval.py').is_file():
        sys.path.insert(0, str(pasta))
        break
from improdutividade_eval import (A, I, P, avaliar_guardrails,
    metricas_seletivas, matriz_2x3)

HOLDOUT_ROOT = Path(os.environ.get('KV_HOLDOUT_DIR', Path.cwd() / 'holdout_final'))
GT_PATH = HOLDOUT_ROOT / 'holdout_ground_truth.csv'
PRED_PATH = HOLDOUT_ROOT / 'holdout_predictions.csv'
DEV_DATES = {'2026-08-24'}  # acrescente aqui todo dia usado em mineração/screening
print('Diretório esperado: holdout_final/ (ou KV_HOLDOUT_DIR)')

Diretório esperado: holdout_final/ (ou KV_HOLDOUT_DIR)


In [2]:
gt_cols = ['unit_id','data','episode_id','video_id','inicio_s','fim_s','peso_s',
           'presenca_true','y_true','anotador_a','anotador_b','adjudicado']
pred_cols = ['candidate_id','unit_id','predicao','evidence_reason','instrument_version']
display(pd.DataFrame(columns=gt_cols).head())
display(pd.DataFrame(columns=pred_cols).head())
if not GT_PATH.is_file() or not PRED_PATH.is_file():
    print('STATUS: BLOQUEADO CORRETAMENTE — os dois arquivos de holdout ainda não existem.')
    print('Use os templates CSV entregues na pasta notebooks/holdout_templates.')
    HOLDOUT_PRONTO = False
else:
    HOLDOUT_PRONTO = True

,unit_id,data,episode_id,video_id,inicio_s,fim_s,peso_s,presenca_true,y_true,anotador_a,anotador_b,adjudicado


,candidate_id,unit_id,predicao,evidence_reason,instrument_version


STATUS: BLOQUEADO CORRETAMENTE — os dois arquivos de holdout ainda não existem.
Use os templates CSV entregues na pasta notebooks/holdout_templates.


In [3]:
if HOLDOUT_PRONTO:
    gt = pd.read_csv(GT_PATH)
    pred = pd.read_csv(PRED_PATH)
    faltam_gt = set(gt_cols) - set(gt.columns)
    faltam_pred = set(pred_cols) - set(pred.columns)
    assert not faltam_gt and not faltam_pred, (faltam_gt, faltam_pred)
    gt['data'] = pd.to_datetime(gt['data'], errors='raise').dt.date.astype(str)
    assert not (set(gt['data']) & DEV_DATES), 'Vazamento: há dia de desenvolvimento no holdout.'
    assert gt['data'].nunique() >= 10, 'Use pelo menos 10 dias independentes.'
    assert gt['episode_id'].notna().all() and gt['episode_id'].nunique() >= 30
    assert gt['unit_id'].is_unique and gt['y_true'].isin([P, I]).all()
    dupla = gt['anotador_b'].notna().mean()
    assert dupla >= 0.20, f'Dupla anotação insuficiente: {dupla:.1%}'
    divergentes = gt['anotador_b'].notna() & (gt['anotador_a'] != gt['anotador_b'])
    assert gt.loc[divergentes, 'adjudicado'].notna().all(), 'Divergência sem adjudicação.'
    esperado = set(gt['unit_id'])
    for cid, bloco in pred.groupby('candidate_id'):
        assert set(bloco['unit_id']) == esperado and bloco['unit_id'].is_unique, f'Cobertura de arquivo incompleta: {cid}'
    aval = pred.merge(gt, on='unit_id', validate='many_to_one')
    print(f'Pré-flight aprovado: {len(gt)} unidades, {gt.data.nunique()} dias, {gt.episode_id.nunique()} episódios, dupla={dupla:.1%}.')

In [4]:
def bootstrap_aninhado(bloco, n=5000, seed=20260912):
    rng = np.random.default_rng(seed)
    dias = list(bloco['data'].unique())
    valores = []
    for _ in range(n):
        partes = []
        for dia in rng.choice(dias, len(dias), replace=True):
            d = bloco[bloco['data'] == dia]
            eps = list(d['episode_id'].unique())
            partes.extend(d[d['episode_id'] == ep] for ep in rng.choice(eps, len(eps), replace=True))
        amostra = pd.concat(partes, ignore_index=True)
        x = metricas_seletivas(amostra, 'predicao')['precision_I']
        if np.isfinite(x): valores.append(x)
    return np.quantile(valores, [0.025, 0.975]) if valores else (np.nan, np.nan)

if HOLDOUT_PRONTO:
    linhas = []
    for cid, bloco in aval.groupby('candidate_id'):
        m = metricas_seletivas(bloco, 'predicao')
        lo, hi = bootstrap_aninhado(bloco)
        linhas.append({'candidate_id': cid, **m, 'precision_I_lcb95': lo, 'precision_I_ucb95': hi})
    resultado = pd.DataFrame(linhas).set_index('candidate_id')
    display(resultado)
    for cid, bloco in aval.groupby('candidate_id'):
        print(f'\nMatriz {cid} em minutos')
        display((matriz_2x3(bloco, 'predicao', 'peso_s') / 60).round(2))

In [5]:
if HOLDOUT_PRONTO:
    assert 'C0' in resultado.index, 'Inclua C0 como baseline congelado.'
    baseline = resultado.loc['C0']
    gates = pd.DataFrame({cid: avaliar_guardrails(linha, baseline)
                          for cid, linha in resultado.iterrows()}).T
    gates['precision_I_LCB>=80%'] = resultado['precision_I_lcb95'] >= 0.80
    gates['aprovada'] = gates.all(axis=1)
    display(gates)
    aprovadas = resultado.loc[gates.index[gates['aprovada']]]
    if aprovadas.empty:
        print('RESULTADO FINAL: nenhuma candidata atingiu o contrato.')
    else:
        vencedora = aprovadas.sort_values(['coverage','recall_I'], ascending=False).index[0]
        print('RESULTADO FINAL: candidata aprovada =', vencedora)
else:
    print('Nenhuma métrica final foi calculada e nenhuma meta foi declarada atingida.')

Nenhuma métrica final foi calculada e nenhuma meta foi declarada atingida.
